In [12]:
# 📊 Nexus GraphRAG — Evaluation Notebook

Compares **Naive RAG** (vector-only Qdrant search + Groq synthesis) vs **Hybrid GraphRAG** (our full ReAct agent API) using [Ragas](https://docs.ragas.io) metrics on 5 complex financial questions from Apple & Microsoft 10-K filings.

| Component | Technology |
|---|---|
| Eval LLM | Groq `llama-3.3-70b-versatile` |
| Naive RAG | Direct Qdrant vector search + Groq answer synthesis |
| Hybrid RAG | `POST http://localhost:8000/ask` (ReAct agent) |
| Metrics | `answer_relevancy`, `answer_correctness` |

> ⚠️ **Before running:** make sure `docker compose up -d` is running so the API is live at `http://localhost:8000`.


SyntaxError: invalid decimal literal (3334107515.py, line 7)

In [11]:

# ── Cell 2: Imports & configuration ──────────────────────────────────────────
import sys, os, time, json, re, textwrap, warnings
import requests
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# Make src/ importable from inside the notebook (no __file__ in Jupyter)
PROJECT_ROOT = Path(os.getcwd()).parent   # evals/ → project root
sys.path.insert(0, str(PROJECT_ROOT))

from src import config

# Ragas 0.4.x — use collections import path to avoid deprecation warnings
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas import evaluate
try:
    from ragas.metrics.collections import answer_relevancy, answer_correctness
except ImportError:
    from ragas.metrics import answer_relevancy, answer_correctness
from datasets import Dataset

# Direct SDKs — no llama-index, Python 3.14 safe
from groq import Groq as GroqClient
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

# ── Validate key ──────────────────────────────────────────────────────────────
assert config.GROQ_API_KEY, "❌ GROQ_API_KEY not set in .env"

GROQ_MODEL   = "llama-3.3-70b-versatile"
EMBED_MODEL  = "BAAI/bge-small-en-v1.5"
COLLECTION   = "sec_10k_filings"
API_URL      = "http://localhost:8000/ask"
GROQ_SLEEP   = 2.0   # seconds between LLM calls (free-tier rate limit)

print("✅ Imports OK")
print(f"   Project root : {PROJECT_ROOT}")
print(f"   Groq model   : {GROQ_MODEL}")
print(f"   API endpoint : {API_URL}")


✅ Imports OK
   Project root : /Users/ramunalla/Personal/Projects/nexus-graphRAG
   Groq model   : llama-3.3-70b-versatile
   API endpoint : http://localhost:8000/ask


In [5]:

# ── Cell 3: Connect to services ───────────────────────────────────────────────
print("Connecting to Groq, Qdrant and loading embedding model…")

groq_client  = GroqClient(api_key=config.GROQ_API_KEY)
qdrant_client = QdrantClient(url=config.QDRANT_URL)
embedder     = TextEmbedding(model_name=EMBED_MODEL)

# Sanity-check the API is reachable
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    health = r.json()
    print(f"✅ API health: {health['status']} | agent: {health['agent']}")
except Exception as e:
    print(f"⚠️  API not reachable ({e}). Make sure 'docker compose up -d' is running.")

print("✅ Qdrant connected | embedding model loaded")


Connecting to Groq, Qdrant and loading embedding model…
✅ API health: operational | agent: ready
✅ Qdrant connected | embedding model loaded


In [6]:

# ── Cell 4: Q&A dataset ───────────────────────────────────────────────────────
qa_pairs = [
    (
        "What major acquisitions did Microsoft make recently, and how did they impact the gaming division?",
        "Microsoft acquired Activision Blizzard, which significantly expanded its gaming division, "
        "adding major franchises and boosting Xbox content and services revenue."
    ),
    (
        "How does Apple rely on third-party manufacturing, and what risks are associated with it?",
        "Apple relies heavily on outsourcing manufacturing to partners primarily in Asia. "
        "Risks include supply chain disruptions, geopolitical tensions, and reliance on single-source suppliers."
    ),
    (
        "What are Microsoft's primary revenue streams within its Intelligent Cloud segment?",
        "The Intelligent Cloud segment revenue is primarily driven by Azure and other cloud services, "
        "server products, and enterprise services."
    ),
    (
        "Who are Apple's main competitors in the smartphone and wearables market?",
        "Apple competes against other global tech companies like Samsung and Google in smartphones, "
        "and various fitness and smart device makers in wearables."
    ),
    (
        "Describe the regulatory risks Microsoft faces regarding data privacy and AI.",
        "Microsoft faces complex regulatory risks including GDPR in Europe, antitrust scrutiny over "
        "its AI partnerships, and emerging AI safety regulations globally."
    ),
]

questions    = [q for q, _ in qa_pairs]
ground_truths = [a for _, a in qa_pairs]

print(f"✅ Dataset ready — {len(questions)} questions loaded")
for i, q in enumerate(questions, 1):
    print(f"  Q{i}: {q[:80]}…")


✅ Dataset ready — 5 questions loaded
  Q1: What major acquisitions did Microsoft make recently, and how did they impact the…
  Q2: How does Apple rely on third-party manufacturing, and what risks are associated …
  Q3: What are Microsoft's primary revenue streams within its Intelligent Cloud segmen…
  Q4: Who are Apple's main competitors in the smartphone and wearables market?…
  Q5: Describe the regulatory risks Microsoft faces regarding data privacy and AI.…


In [7]:

# ── Cell 5: Naive RAG helper (direct Qdrant + Groq, no llama-index) ───────────
def naive_rag_answer(question: str) -> str:
    """
    Naive RAG baseline:
      1. Embed the question with fastembed
      2. Retrieve top-5 chunks from Qdrant
      3. Ask Groq to answer using only those chunks
    """
    # Step 1 — embed
    q_vector = list(embedder.embed([question]))[0].tolist()

    # Step 2 — retrieve (qdrant-client 1.7+ uses query_points)
    try:
        response = qdrant_client.query_points(
            collection_name=COLLECTION,
            query=q_vector,
            limit=5,
        )
        hits = response.points
    except AttributeError:
        hits = qdrant_client.search(
            collection_name=COLLECTION,
            query_vector=q_vector,
            limit=5,
        )

    if not hits:
        return "No relevant context found in the vector store."

    context = "\n\n---\n\n".join(
        f"[{h.payload.get('source','?')}] {h.payload.get('text','')}" for h in hits
    )

    # Step 3 — generate answer
    prompt = textwrap.dedent(f"""\
        Answer the question using ONLY the provided context. Be concise and factual.

        Context:
        {context}

        Question: {question}
        Answer:""")

    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()


print("✅ naive_rag_answer() defined")


✅ naive_rag_answer() defined


In [8]:

# ── Cell 6: Run both systems against all questions ────────────────────────────
naive_answers  = []
hybrid_answers = []

for i, question in enumerate(questions, 1):
    print(f"\n{'─'*60}")
    print(f"Q{i}: {question}")

    # ── A: Naive RAG ──────────────────────────────────────────
    print("  [Naive RAG] querying…", end=" ", flush=True)
    try:
        naive_ans = naive_rag_answer(question)
        naive_answers.append(naive_ans)
        print(f"done ✅  ({len(naive_ans)} chars)")
    except Exception as e:
        naive_answers.append(f"Error: {e}")
        print(f"❌ {e}")

    time.sleep(GROQ_SLEEP)

    # ── B: Hybrid GraphRAG API ─────────────────────────────────
    print("  [Hybrid GraphRAG] querying…", end=" ", flush=True)
    try:
        api_resp = requests.post(
            API_URL,
            json={"question": question},
            timeout=120,
        )
        if api_resp.status_code == 200:
            hybrid_ans = api_resp.json().get("answer", "No answer field")
            hybrid_answers.append(hybrid_ans)
            print(f"done ✅  ({len(hybrid_ans)} chars)")
        else:
            hybrid_answers.append(f"API HTTP {api_resp.status_code}")
            print(f"❌ HTTP {api_resp.status_code}")
    except Exception as e:
        hybrid_answers.append(f"API Connection Error: {e}")
        print(f"❌ {e}")

    time.sleep(GROQ_SLEEP)

print(f"\n\n✅ All {len(questions)} questions answered by both systems.")



────────────────────────────────────────────────────────────
Q1: What major acquisitions did Microsoft make recently, and how did they impact the gaming division?
  [Naive RAG] querying… done ✅  (131 chars)
  [Hybrid GraphRAG] querying… done ✅  (55 chars)

────────────────────────────────────────────────────────────
Q2: How does Apple rely on third-party manufacturing, and what risks are associated with it?
  [Naive RAG] querying… done ✅  (266 chars)
  [Hybrid GraphRAG] querying… done ✅  (363 chars)

────────────────────────────────────────────────────────────
Q3: What are Microsoft's primary revenue streams within its Intelligent Cloud segment?
  [Naive RAG] querying… done ✅  (546 chars)
  [Hybrid GraphRAG] querying… done ✅  (282 chars)

────────────────────────────────────────────────────────────
Q4: Who are Apple's main competitors in the smartphone and wearables market?
  [Naive RAG] querying… done ✅  (293 chars)
  [Hybrid GraphRAG] querying… done ✅  (55 chars)

──────────────────

In [ ]:

# ── Cell 7: Ragas evaluation ──────────────────────────────────────────────────
# Python 3.14 broke nest_asyncio. ragas.evaluate() calls asyncio.run() which
# nest_asyncio patches — causing a crash. Fix: run ragas as a subprocess so it
# gets a completely clean Python process with no Jupyter event loop interference.

import subprocess, sys, json, tempfile, os
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import BaseRagasEmbeddings


def run_ragas_subprocess(questions, answers, ground_truths, groq_api_key, groq_model):
    """
    Serialize data, spawn a clean Python subprocess to run ragas,
    return the results dict. Fully isolated from the Jupyter event loop.
    """
    script = f"""
import asyncio, warnings, json, sys
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import BaseRagasEmbeddings
from ragas.run_config import RunConfig
from ragas.metrics import answer_relevancy, answer_correctness
from ragas import evaluate
from datasets import Dataset
from fastembed import TextEmbedding

class FastEmbedRagas(BaseRagasEmbeddings):
    def __init__(self):
        super().__init__()
        self.run_config = RunConfig()
        self._m = TextEmbedding("BAAI/bge-small-en-v1.5")
    def embed_query(self, t): return list(self._m.embed([t]))[0].tolist()
    def embed_documents(self, ts): return [v.tolist() for v in self._m.embed(ts)]
    async def aembed_query(self, t): return self.embed_query(t)
    async def aembed_documents(self, ts): return self.embed_documents(ts)

questions    = {json.dumps(questions)}
answers      = {json.dumps(answers)}
ground_truths = {json.dumps(ground_truths)}

llm  = LangchainLLMWrapper(ChatGroq(model="{groq_model}", api_key="{groq_api_key}", temperature=0))
emb  = FastEmbedRagas()

ds = Dataset.from_dict({{"question": questions, "answer": answers, "ground_truth": ground_truths}})
r  = evaluate(ds, metrics=[answer_relevancy, answer_correctness], llm=llm, embeddings=emb)
df = r.to_pandas()
out = {{
    "answer_relevancy":   float(df["answer_relevancy"].mean()),
    "answer_correctness": float(df["answer_correctness"].mean()),
}}
print(json.dumps(out))
"""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(script)
        tmp = f.name

    try:
        result = subprocess.run(
            [sys.executable, tmp],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0:
            raise RuntimeError(f"Ragas subprocess failed:\n{result.stderr[-2000:]}")
        # Last line of stdout is the JSON result
        last_line = [l for l in result.stdout.strip().splitlines() if l.strip()][-1]
        return json.loads(last_line)
    finally:
        os.unlink(tmp)


print("Running Ragas on Naive RAG answers (subprocess)…")
naive_scores = run_ragas_subprocess(
    questions, naive_answers, ground_truths,
    config.GROQ_API_KEY, GROQ_MODEL
)
print(f"   ✅ Naive  → {naive_scores}")

print("Running Ragas on Hybrid GraphRAG answers (subprocess)…")
hybrid_scores = run_ragas_subprocess(
    questions, hybrid_answers, ground_truths,
    config.GROQ_API_KEY, GROQ_MODEL
)
print(f"   ✅ Hybrid → {hybrid_scores}")

print("✅ Ragas evaluation complete")


Initialising judge LLM and fastembed wrapper…
Running Ragas on Naive RAG answers…


Exception in callback _patch_task.<locals>.step()
handle: <Handle _patch_task.<locals>.step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/nest_asyncio.py", line 205, in step
    step_orig(task, exc)
    ~~~~~~~~~^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/tasks.py", line 276, in __step
    _py_enter_task(self._loop, self)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/tasks.py", line 1083, in _enter_task
    raise RuntimeError(f"Cannot enter into task {task!r} while another "
                       f"task {current_task!r} is being executed.")
RuntimeError: Cannot enter into task